## Transformation and Claen up of the dataset 
In this section we are going to clean and tudy up the data after doing the exploration. We will drop some columns and encode a few so lets dive into it

In [43]:
#importing libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [44]:
#reading the dataset
raw_data = pd.read_csv("/Users/sa03/Fin_Fraud_Detection/Dataset/PS_20174392719_1491204439457_log.csv")   

#checking the shape of the dataset
raw_data.shape

(6362620, 11)

We can see that our dataset is over 6 million which is too large so we will generate a sample from the population and perform the rest of our analysis on.

In [56]:
#creating a gitignore file for the dataset
# this will prevent the dataset from being pushed to the remote repository
with open('.gitignore', 'w') as f:
    f.write('/Users/sa03/Fin_Fraud_Detection/Dataset/PS_20174392719_1491204439457_log.csv\n')


In [46]:
#df = pd.read_csv("/Users/sa03/Fin_Fraud_Detection/Dataset/PS_20174392719_1491204439457_log.csv") --- IGNORE ---
# --- IGNORE ---    

In [47]:
#Generating a random sample of 180000 rows from the dataset for EDA and transformation

df = raw_data.sample(n=180000, random_state=42)


In [48]:
#checking the information about the dataset

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 180000 entries, 3737323 to 4003207
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   step            180000 non-null  int64  
 1   type            180000 non-null  object 
 2   amount          180000 non-null  float64
 3   nameOrig        180000 non-null  object 
 4   oldbalanceOrg   180000 non-null  float64
 5   newbalanceOrig  180000 non-null  float64
 6   nameDest        180000 non-null  object 
 7   oldbalanceDest  180000 non-null  float64
 8   newbalanceDest  180000 non-null  float64
 9   isFraud         180000 non-null  int64  
 10  isFlaggedFraud  180000 non-null  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 16.5+ MB


In [49]:
#Dropping the columns which are not required for the analysis
df.drop(columns=['nameOrig','nameDest','isFlaggedFraud'])

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
3737323,278,CASH_IN,330218.42,20866.00,351084.42,452419.57,122201.15,0
264914,15,PAYMENT,11647.08,30370.00,18722.92,0.00,0.00,0
85647,10,CASH_IN,152264.21,106589.00,258853.21,201303.01,49038.80,0
5899326,403,TRANSFER,1551760.63,0.00,0.00,3198359.45,4750120.08,0
2544263,206,CASH_IN,78172.30,2921331.58,2999503.88,415821.90,337649.60,0
...,...,...,...,...,...,...,...,...
660411,36,CASH_IN,304551.75,10820.00,315371.75,622141.59,317589.84,0
1977994,179,PAYMENT,1358.83,0.00,0.00,0.00,0.00,0
3763755,279,PAYMENT,7071.54,11294.00,4222.46,0.00,0.00,0
1729405,160,CASH_IN,34861.56,3406931.77,3441793.33,620241.35,585379.78,0


We are dropping all thses columns because they are not useful to our model and also something like nameOrig is just a random username and has no specfic or useful pattern in it

In [50]:
# We can see that the "type" column is of object data type. 

df['type']

3737323     CASH_IN
264914      PAYMENT
85647       CASH_IN
5899326    TRANSFER
2544263     CASH_IN
             ...   
660411      CASH_IN
1977994     PAYMENT
3763755     PAYMENT
1729405     CASH_IN
4003207     CASH_IN
Name: type, Length: 180000, dtype: object

In [51]:
# let us encode the "type" column using one hot encoding

df = pd.get_dummies(df, columns=['type'], drop_first=True)


In [52]:
# Let us check the first 5 rows of the dataset after encoding

df.head()

,step,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
3737323,278,330218.42,C632336343,20866.00,351084.42,C834976624,452419.57,122201.15,0,0,False,False,False,False
264914,15,11647.08,C1264712553,30370.00,18722.92,M215391829,0.00,0.00,0,0,False,False,True,False
85647,10,152264.21,C1746846248,106589.00,258853.21,C1607284477,201303.01,49038.80,0,0,False,False,False,False
5899326,403,1551760.63,C333676753,0.00,0.00,C1564353608,3198359.45,4750120.08,0,0,False,False,False,True
2544263,206,78172.30,C813403091,2921331.58,2999503.88,C1091768874,415821.90,337649.60,0,0,False,False,False,False


## Feature engineer a few fraud signals 

In [54]:
# How much did our original balance change

df['balance_change'] = df['oldbalanceOrg'] - df['newbalanceOrig']

print(df['balance_change'].head())

3737323   -330218.42
264914      11647.08
85647     -152264.21
5899326         0.00
2544263    -78172.30
Name: balance_change, dtype: float64


In [55]:
# How much did the destination balance change

df['dest_balance_change'] = df['oldbalanceDest'] - df['newbalanceDest']

print(df['dest_balance_change'].head())

3737323     330218.42
264914           0.00
85647       152264.21
5899326   -1551760.63
2544263      78172.30
Name: dest_balance_change, dtype: float64


In [39]:
#Lets check to see if there are any missing values in the dataset

df.isnull().sum()

step              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
type_CASH_OUT     0
type_DEBIT        0
type_PAYMENT      0
type_TRANSFER     0
dtype: int64

We see our dataset looks very clean now with no null values and the columns we will not be needing for our model dropped as well 

In [42]:
# Now let us save our transformed dataset to a new csv file

df.to_csv('transformed_data.csv', index=False)

print("Saved! Shape:", df.shape)

Saved! Shape: (180000, 14)
